# Silver Layer — Dimensiuni simple

**Scop:** Construirea celor 5 dimensiuni "simple" (fără SCD Type 2):
`dim_branches`, `dim_employees`, `dim_accounts`, `dim_cards`, `dim_loans`.

**`dim_customers` cu SCD Type 2** se face în notebook-ul separat `02b2`.

**Pentru fiecare dimensiune:**
1. Citire din `banking.bronze.raw_*`
2. Type casting (STRING → INTEGER, DOUBLE, DATE, TIMESTAMP)
3. Validări de calitate → înregistrările invalide merg în `banking.silver_quarantine.qrt_*`
4. Înregistrările valide se scriu în `banking.silver.dim_*`
5. Păstrăm metadata de lineage din Bronze (`source_batch_id`, `source_ingestion_ts`)

**Validări per dimensiune:**
- **`dim_branches`**: PK non-null, county_code/region_code valide
- **`dim_employees`**: PK non-null, branch_id valid (FK), salariu pozitiv
- **`dim_accounts`**: PK non-null, customer_id/branch_id valide, balance numeric
- **`dim_cards`**: PK non-null, account_id valid, expiry_date format, **detectare carduri expirate cu status=ACTIVE**
- **`dim_loans`**: PK non-null, account_id valid, **detectare end_date < start_date**, sume pozitive

## 1. Configurare și utilitare

In [0]:
from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, when, lit, current_timestamp, to_date, to_timestamp,
    array, expr, concat_ws, size,
    filter as array_filter
)
from pyspark.sql.types import IntegerType, DoubleType, StringType

CATALOG     = "banking"
BRONZE      = "bronze"
SILVER      = "silver"
QUARANTINE  = "silver_quarantine"

print(f"Bronze:     {CATALOG}.{BRONZE}.*")
print(f"Silver:     {CATALOG}.{SILVER}.dim_*")
print(f"Quarantine: {CATALOG}.{QUARANTINE}.qrt_*")

Bronze:     banking.bronze.*
Silver:     banking.silver.dim_*
Quarantine: banking.silver_quarantine.qrt_*


## 2. Funcții helper comune

In [0]:
def read_bronze(table_short_name: str) -> DataFrame:
    """Citeste un tabel din Bronze (raw_<name>) si pastreaza lineage metadata."""
    df = spark.table(f"{CATALOG}.{BRONZE}.raw_{table_short_name}")
    # Pastram metadata Bronze redenumita pentru lineage end-to-end
    if "_batch_id" in df.columns:
        df = df.withColumnRenamed("_batch_id", "source_batch_id")
    if "_ingestion_ts" in df.columns:
        df = df.withColumnRenamed("_ingestion_ts", "source_ingestion_ts")
    
    # Eliminam metadatele duplicate
    for c in ["_source_system", "_source_file"]:
        if c in df.columns:
            df = df.drop(c)
    
    return df


def cast_columns(df: DataFrame, casts: dict) -> DataFrame:
    """
    Aplica type casting pe coloane.
    casts: {"col_name": IntegerType()} etc.
    """
    for col_name, target_type in casts.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, col(col_name).cast(target_type))
    return df


def split_valid_invalid(
    df: DataFrame, 
    validations: list
) -> tuple[DataFrame, DataFrame]:
    """
    Aplica o lista de validari pe DataFrame.
    
    validations: lista de tupluri (rule_name, condition_expression)
        - condition_expression = TRUE inseamna 'rand valid'
    
    Returneaza (df_valid, df_invalid).
    df_invalid contine coloana extra 'error_reasons' (array de strings).
    """
    from pyspark.sql.functions import filter as array_filter, expr
    
    # Construim array-ul cu nume rule + flag
    # Pentru fiecare regula adaugam o coloana auxiliara cu numele regulii daca e incalcata
    error_cols = []
    for rule_name, condition in validations:
        col_name = f"_err_{rule_name}"
        df = df.withColumn(col_name, when(~condition, lit(rule_name)))
        error_cols.append(col_name)
    
    # Construim array-ul cu erorile, apoi filtram NULL-urile cu higher-order function filter()
    df = df.withColumn(
        "error_reasons",
        array_filter(array(*[col(c) for c in error_cols]), lambda x: x.isNotNull())
    )
    
    # Cleanup coloane auxiliare
    df = df.drop(*error_cols)
    
    df_valid   = df.filter(size(col("error_reasons")) == 0).drop("error_reasons")
    df_invalid = df.filter(size(col("error_reasons")) > 0)
    
    return df_valid, df_invalid


def write_silver(df: DataFrame, table_name: str):
    """Scrie un DataFrame in Silver cu metadata Silver."""
    df_final = df.withColumn("silver_loaded_at", current_timestamp())
    (df_final.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SILVER}.{table_name}"))


def write_quarantine(df: DataFrame, table_name: str):
    """Scrie un DataFrame de invalide in Quarantine cu metadata."""
    df_final = (df
        .withColumn("quarantine_loaded_at", current_timestamp())
        .withColumn("error_reasons_str", concat_ws(", ", col("error_reasons")))
    )
    (df_final.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{QUARANTINE}.qrt_{table_name}"))


def report_split(name: str, df_valid: DataFrame, df_invalid: DataFrame):
    """Print sumar split valid/invalid."""
    n_valid   = df_valid.count()
    n_invalid = df_invalid.count()
    n_total   = n_valid + n_invalid
    pct       = (n_invalid / n_total * 100) if n_total > 0 else 0
    print(f"  {name:20s} valid={n_valid:>6d}  invalid={n_invalid:>4d}  ({pct:.1f}% quarantine)")

## 3. dim_branches

Sucursalele sunt simple — doar type casting și validări de bază.

In [0]:
print("Procesare dim_branches...")

df = read_bronze("branches")

# Type casting
df = cast_columns(df, {
    "is_active": IntegerType(),
})

# Conversii date
df = df.withColumn("opened_date", to_date(col("opened_date"), "yyyy-MM-dd"))

# Validari
validations = [
    ("pk_null",          col("branch_id").isNotNull()),
    ("name_null",        col("name").isNotNull()),
    ("county_code_null", col("county_code").isNotNull()),
    ("region_code_null", col("region_code").isNotNull()),
]

df_valid, df_invalid = split_valid_invalid(df, validations)
report_split("dim_branches", df_valid, df_invalid)

write_silver(df_valid, "dim_branches")
if df_invalid.count() > 0:
    write_quarantine(df_invalid, "branches")

Procesare dim_branches...
  dim_branches         valid=    20  invalid=   0  (0.0% quarantine)


## 4. dim_employees

In [0]:
print("Procesare dim_employees...")

df = read_bronze("employees")

df = cast_columns(df, {
    "salary": DoubleType(),
})
df = df.withColumn("hire_date", to_date(col("hire_date"), "yyyy-MM-dd"))

# Pentru validare FK branch_id, citim sucursalele valide din Silver
valid_branch_ids = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.dim_branches")
                    .select("branch_id").collect()]

validations = [
    ("pk_null",            col("employee_id").isNotNull()),
    ("branch_id_null",     col("branch_id").isNotNull()),
    ("branch_id_invalid",  col("branch_id").isin(valid_branch_ids)),
    ("first_name_null",    col("first_name").isNotNull()),
    ("salary_negative",    (col("salary").isNull()) | (col("salary") >= 0)),
    ("status_invalid",     col("status").isin(["ACTIVE", "ON_LEAVE", "TERMINATED"])),
]

df_valid, df_invalid = split_valid_invalid(df, validations)
report_split("dim_employees", df_valid, df_invalid)

write_silver(df_valid, "dim_employees")
if df_invalid.count() > 0:
    write_quarantine(df_invalid, "employees")

Procesare dim_employees...
  dim_employees        valid=    60  invalid=   0  (0.0% quarantine)


## 5. dim_accounts

In [0]:
print("Procesare dim_accounts...")

df = read_bronze("accounts")

df = cast_columns(df, {
    "balance":           DoubleType(),
    "available_balance": DoubleType(),
})
df = df.withColumn("open_date",  to_date(col("open_date"),  "yyyy-MM-dd"))
df = df.withColumn("close_date", to_date(col("close_date"), "yyyy-MM-dd"))

# FK references
valid_branch_ids = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.dim_branches")
                    .select("branch_id").collect()]

# Pentru customers, lasam validarea FK pe seama tabelei dim_customers (creata in 02b2)
# Aici verificam doar ca customer_id e populat

validations = [
    ("pk_null",            col("account_id").isNotNull()),
    ("customer_id_null",   col("customer_id").isNotNull()),
    ("branch_id_null",     col("branch_id").isNotNull()),
    ("branch_id_invalid",  col("branch_id").isin(valid_branch_ids)),
    ("balance_null",       col("balance").isNotNull()),
    ("status_invalid",     col("status").isin(["ACTIVE", "FROZEN", "BLOCKED", "CLOSED"])),
    ("close_date_logic",   (col("close_date").isNull()) | (col("close_date") >= col("open_date"))),
    ("iban_null",          col("iban").isNotNull()),
]

df_valid, df_invalid = split_valid_invalid(df, validations)
report_split("dim_accounts", df_valid, df_invalid)

write_silver(df_valid, "dim_accounts")
if df_invalid.count() > 0:
    write_quarantine(df_invalid, "accounts")

Procesare dim_accounts...
  dim_accounts         valid=   800  invalid=   0  (0.0% quarantine)


## 6. dim_cards

**Validare critică:** detectăm cardurile **expirate cu status ACTIVE** (eroare logică injectată în generator).

In [0]:
print("Procesare dim_cards...")

df = read_bronze("cards")

df = cast_columns(df, {
    "credit_limit":    DoubleType(),
    "current_balance": DoubleType(),
})
df = df.withColumn("issued_date", to_date(col("issued_date"), "yyyy-MM-dd"))

# expiry_date e in format YYYY-MM (luna calendaristica) — convertim la prima zi a lunii
df = df.withColumn(
    "expiry_date",
    to_date(concat_ws("-", col("expiry_date"), lit("01")), "yyyy-MM-dd")
)

# FK
valid_account_ids = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.dim_accounts")
                     .select("account_id").collect()]

today = current_timestamp().cast("date")

validations = [
    ("pk_null",                col("card_id").isNotNull()),
    ("account_id_null",        col("account_id").isNotNull()),
    ("account_id_invalid",     col("account_id").isin(valid_account_ids)),
    ("status_invalid",         col("status").isin(["ACTIVE", "BLOCKED", "EXPIRED", "CANCELLED"])),
    ("expiry_date_null",       col("expiry_date").isNotNull()),
    ("issued_date_null",       col("issued_date").isNotNull()),
    ("expired_but_active",     ~((col("status") == "ACTIVE") & (col("expiry_date") < today))),
    ("credit_limit_negative",  (col("credit_limit").isNull()) | (col("credit_limit") >= 0)),
]

df_valid, df_invalid = split_valid_invalid(df, validations)
report_split("dim_cards", df_valid, df_invalid)

write_silver(df_valid, "dim_cards")
if df_invalid.count() > 0:
    write_quarantine(df_invalid, "cards")

Procesare dim_cards...
  dim_cards            valid=   586  invalid=  14  (2.3% quarantine)


## 7. dim_loans

**Validare critică:** detectăm credite cu **end_date < start_date** (eroare logică injectată).

In [0]:
print("Procesare dim_loans...")

df = read_bronze("loans")

df = cast_columns(df, {
    "principal_amount":    DoubleType(),
    "outstanding_balance": DoubleType(),
    "interest_rate_pa":    DoubleType(),
    "monthly_payment":     DoubleType(),
    "term_months":         IntegerType(),
})
df = df.withColumn("start_date",  to_date(col("start_date"),  "yyyy-MM-dd"))
df = df.withColumn("end_date",    to_date(col("end_date"),    "yyyy-MM-dd"))
df = df.withColumn("approved_at", to_timestamp(col("approved_at")))

# FK
valid_account_ids = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.dim_accounts")
                     .select("account_id").collect()]

validations = [
    ("pk_null",                  col("loan_id").isNotNull()),
    ("account_id_null",          col("account_id").isNotNull()),
    ("account_id_invalid",       col("account_id").isin(valid_account_ids)),
    ("status_invalid",           col("status").isin(
        ["APPLIED", "APPROVED", "ACTIVE", "OVERDUE", "CLOSED", "REJECTED"])),
    ("principal_invalid",        col("principal_amount") > 0),
    ("end_date_before_start",    col("end_date") >= col("start_date")),
    ("term_invalid",             col("term_months") > 0),
    ("rate_negative",            col("interest_rate_pa") >= 0),
    ("outstanding_invalid",      col("outstanding_balance") >= 0),
]

df_valid, df_invalid = split_valid_invalid(df, validations)
report_split("dim_loans", df_valid, df_invalid)

write_silver(df_valid, "dim_loans")
if df_invalid.count() > 0:
    write_quarantine(df_invalid, "loans")

Procesare dim_loans...
  dim_loans            valid=   195  invalid=   5  (2.5% quarantine)


## 8. Verificare — listare dimensiuni create

In [0]:
%sql
SHOW TABLES IN banking.silver;

database,tableName,isTemporary
silver,dim_accounts,false
silver,dim_branches,false
silver,dim_cards,false
silver,dim_employees,false
silver,dim_loans,false
silver,ref_account_types,false
silver,ref_card_types,false
silver,ref_channels,false
silver,ref_counties,false
silver,ref_countries,false


## 9. Verificare carantină

In [0]:
%sql
SHOW TABLES IN banking.silver_quarantine;

database,tableName,isTemporary
silver_quarantine,qrt_cards,false
silver_quarantine,qrt_loans,false
,_sqldf,true


## 10. Inspecție quarantine — exemple de erori detectate

In [0]:
%sql
-- Exemple de carduri expirate dar status=ACTIVE (eroare logica injectata)
SELECT 
    card_id, 
    account_id, 
    status, 
    expiry_date, 
    error_reasons_str
FROM banking.silver_quarantine.qrt_cards
WHERE array_contains(error_reasons, 'expired_but_active')
LIMIT 10;

card_id,account_id,status,expiry_date,error_reasons_str
CARD-000017,ACC-000727,ACTIVE,2025-05-01,expired_but_active
CARD-000036,ACC-000542,ACTIVE,2025-12-01,expired_but_active
CARD-000133,ACC-000205,ACTIVE,2025-05-01,expired_but_active
CARD-000181,ACC-000451,ACTIVE,2025-06-01,expired_but_active
CARD-000200,ACC-000135,ACTIVE,2025-08-01,expired_but_active
CARD-000208,ACC-000272,ACTIVE,2025-12-01,expired_but_active
CARD-000276,ACC-000144,ACTIVE,2025-10-01,expired_but_active
CARD-000374,ACC-000371,ACTIVE,2025-06-01,expired_but_active
CARD-000385,ACC-000191,ACTIVE,2025-12-01,expired_but_active
CARD-000477,ACC-000059,ACTIVE,2025-09-01,expired_but_active


In [0]:
%sql
-- Exemple de credite cu end_date < start_date (eroare logica injectata)
SELECT 
    loan_id, 
    account_id, 
    start_date, 
    end_date, 
    error_reasons_str
FROM banking.silver_quarantine.qrt_loans
WHERE array_contains(error_reasons, 'end_date_before_start')
LIMIT 10;

loan_id,account_id,start_date,end_date,error_reasons_str
LOAN-000003,ACC-000535,2025-10-02,2025-06-30,end_date_before_start
LOAN-000039,ACC-000252,2025-02-10,2024-11-10,end_date_before_start
LOAN-000162,ACC-000189,2021-05-12,2021-01-02,end_date_before_start
LOAN-000168,ACC-000383,2024-10-01,2024-05-30,end_date_before_start
LOAN-000180,ACC-000434,2022-03-15,2021-10-29,end_date_before_start


## 11. Sumar global — câte rânduri în Silver vs Quarantine

In [0]:
%sql
SELECT 
    'dim_branches'  AS dim, 
    (SELECT COUNT(*) FROM banking.silver.dim_branches)  AS valid_rows,
    (SELECT COUNT(*) FROM banking.bronze.raw_branches)  AS bronze_rows
UNION ALL SELECT 'dim_employees',
    (SELECT COUNT(*) FROM banking.silver.dim_employees),
    (SELECT COUNT(*) FROM banking.bronze.raw_employees)
UNION ALL SELECT 'dim_accounts',
    (SELECT COUNT(*) FROM banking.silver.dim_accounts),
    (SELECT COUNT(*) FROM banking.bronze.raw_accounts)
UNION ALL SELECT 'dim_cards',
    (SELECT COUNT(*) FROM banking.silver.dim_cards),
    (SELECT COUNT(*) FROM banking.bronze.raw_cards)
UNION ALL SELECT 'dim_loans',
    (SELECT COUNT(*) FROM banking.silver.dim_loans),
    (SELECT COUNT(*) FROM banking.bronze.raw_loans);

dim,valid_rows,bronze_rows
dim_branches,20,20
dim_employees,60,60
dim_accounts,800,800
dim_cards,586,600
dim_loans,195,200


## Sumar 02b1 — Silver Dimensions (simple)

Realizat:
- 5 dimensiuni Silver create cu type casting corect
- Validări FK aplicate (branch_id în dim_employees/dim_accounts, account_id în dim_cards/dim_loans)
- Erori logice detectate și mutate în quarantine:
  - `dim_cards`: carduri expirate cu status ACTIVE
  - `dim_loans`: end_date înainte de start_date
- Lineage păstrat: `source_batch_id`, `source_ingestion_ts`, `silver_loaded_at`

**Următorul pas:** `02b2_silver_dim_customers_scd2` — implementarea SCD Type 2 pentru `dim_customers` cu trackare `segment_code`, `kyc_status`, `county_code`, `is_active`.